In [ ]:
## PROCESS FOR RUNNING ALL PARTICIPANTS FOR EACH STUDY
## FOR EACH STUDY
## 1) Specify sleep schedule
## 2) Get Inidivual data
## 3) Run Parameter sweep for timepoint X
## 5) Run parameter sweep again with best set from previous sweep
## 6) Compare distance between new params and old (and/or 
## 7) Iterate through 5 and 6 until distance or KL val does not diminish

In [ ]:
library(Hmisc)
library(haven)
library(utils)
library(jsonlite)
library(stats)
#library(likelihood)
#library(GenSA)
library(optimization)
library(DBI)
library(RPostgreSQL)
library(dbplyr)
library(ks)
library(philentropy)
library(doBy)
library(lme4)
library(cluster)
library(mclust)
library(randomForest)
library(rstatix)
library(effectsize)
library(tidyverse)
library(lubridate)
library(furrr)
library(future)
library(dplyr)
library(purrr)
library(GGally)
library(ggpubr)
library(patchwork)
library(scales)
library(ggh4x)
library(viridis)
library(scico)
library(reshape2)
library(grid)
library(ggrepel)
library(ggallin)
library(ggnewscale)
library(RColorBrewer)
library(GGally)
library(ggpubr)
library(patchwork)
#library(broom)

# Load and Process Human Data

## Load OPSI and OPSII Data

# Get Sleep Data

In [ ]:
ops2_sleep_data <- read_csv("ops2_sleep_data.csv",show_col_types = FALSE)
head(ops2_sleep_data)

In [ ]:
high_sleep_data <- read_csv("high_sleep_data.csv",show_col_types = FALSE)
head(high_sleep_data)

# Get Behavioral Data

## OPSI and OPSII Data

In [ ]:
nback_ops1_data_cleaned <- read_csv("nback_ops1_data.csv",show_col_types = FALSE)
head(nback_ops1_data_cleaned)

In [ ]:
pvt_ops1_data_cleaned <- read_csv("pvt_ops1_data.csv",show_col_types = FALSE)
head(pvt_ops1_data_cleaned)

In [ ]:
nback_ops2_data_cleaned <- read_csv("nback_ops2_data.csv",show_col_types = FALSE)
head(nback_ops2_data_cleaned)

nback_ops2_data_cleaned <- nback_ops2_data_cleaned %>% filter(study_event_id %in% c("6","22","38")) %>% group_by(subjectid) %>%
    filter(length(unique(study_event_id)) == 3) #%>%
    #group_by(subjectid, study_event_id) %>%
    #filter(sum(!is.na(ResponseTime)) != 0) %>%
    #group_by(subjectid) %>% filter(n_distinct(phase) == 3)

head(nback_ops2_data_cleaned)
unique(nback_ops2_data_cleaned$subjectid)
n_distinct(nback_ops2_data_cleaned$subjectid)

In [ ]:
pvt_ops2_data_cleaned <- read_csv("pvt_ops2_data.csv",show_col_types = FALSE)
head(pvt_ops2_data_cleaned)

pvt_ops2_data_cleaned <- pvt_ops2_data_cleaned %>% filter(study_event_id %in% c("6","22","38")) %>% group_by(subjectid) %>%
    filter(length(unique(study_event_id)) == 3) %>%
    group_by(subjectid, study_event_id) %>%
    filter(sum(!is.na(RespTime)) != 0) %>%
    group_by(subjectid) %>% filter(n_distinct(phase) == 3)

n_distinct(pvt_ops2_data_cleaned$subjectid)
head(pvt_ops2_data_cleaned)

## SERE DATA

In [ ]:
pvt_sere_data_cleaned <- read_csv("pvt_sere_data.csv",show_col_types = FALSE)
head(pvt_sere_data_cleaned)

pvt_sere_data_cleaned <- pvt_sere_data_cleaned %>% filter(study_event_id %in% c("bt","pt","rt")) %>% group_by(subjectid) %>%
    filter(length(unique(study_event_id)) == 3) %>%
    group_by(subjectid, study_event_id) %>%
    filter(sum(!is.na(RespTime)) != 0) %>%
    group_by(subjectid) %>% filter(n_distinct(phase) == 3)

n_distinct(pvt_sere_data_cleaned$subjectid)
head(pvt_sere_data_cleaned)

## HIGH Data

In [ ]:
nback_high_data_cleaned <- read_csv("nback_high_data.csv",show_col_types = FALSE)
head(nback_high_data_cleaned)


nback_high_data_cleaned <- nback_high_data_cleaned %>% filter(study_event_id %in% c("B1",
                                                                                    #"A1","A2",
                                                                                    "A3")) %>% group_by(subjectid) %>%
    filter(length(unique(study_event_id)) == 2) #4) #%>%
    #group_by(subjectid, study_event_id) %>%
    #filter(sum(!is.na(ResponseTime)) != 0) %>%
    #group_by(subjectid) %>% filter(n_distinct(phase) == 2) %>%
    #filter(length(unique(study_event_id)) == 4)

unique(nback_high_data_cleaned$subjectid)
n_distinct(nback_high_data_cleaned$subjectid)
head(nback_high_data_cleaned)

In [ ]:
pvt_high_data_cleaned <- read_csv("pvt_high_data.csv",show_col_types = FALSE)
head(pvt_high_data_cleaned)

pvt_high_data_cleaned <- pvt_high_data_cleaned %>% filter(study_event_id %in% c("B1",
                                                                                #"A1","A2",
                                                                                "A3")) %>% group_by(subjectid) %>%
    filter(length(unique(study_event_id)) == 2) %>% #4) %>%
    group_by(subjectid, study_event_id) %>%
    filter(sum(!is.na(RespTime)) != 0) %>%
    group_by(subjectid) %>% filter(n_distinct(phase) == 2) %>%
    filter(length(unique(study_event_id)) == 2) #4)

unique(pvt_high_data_cleaned$subjectid)
n_distinct(pvt_high_data_cleaned$subjectid)
head(pvt_high_data_cleaned)

# PVT PARAMETER OPTIMISATION FUNCTIONS

In [ ]:
########################   START CREATE MODEL FUNCTION   ###########################
## Create our model function
## Send message to model as json of key:value pairs
## where key = parameter name and value = parameter value
run_model <- function(study,sched,bout,params,iterations,PORT) {
    #startTime <- Sys.time()
    #cat(sprintf("# Start Time: %s\n",startTime))              
    
    params <- list(dat = params[1], egs = params[2], ut = params[3], iu = params[4], utmc = params[5], utbmc = params[6], fpdec = params[7], fpmc = params[8], fpbmc = params[9], bouts = list(bout), schedule = sched, numiterations = iterations)
    cat(sprintf("# request '%s'\n", toJSON(params)))
    params <- toJSON(params, auto_unbox = TRUE)
    cat(sprintf("# sending '%s'\n", params))

    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    sock <- make.socket(host = HOST, port = PORT, server = FALSE)
    on.exit(close.socket(sock))
    
    # Connect to server and send data
    write.socket(sock, paste(params,"\r\n"))
    cat(sprintf("# connected to port '%s'\n", PORT))

    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    # Receive data from the server and shut down
    #received <- character(0)
    received <- character()
    repeat {
        ss <- read.socket(sock)
        if (ss == "") break
        received <- c(received, ss)
    }
    
    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    received <- paste(received, collapse = "")

    cat(sprintf("# received message of length %s\n", nchar(received)))
    raw_data <- fromJSON(fromJSON(received))

    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    ############## EXAMPLE received DATA FRAME ###############
    ##########################################################
    ##       iteration	day	    bout	curtime	delay	rt
    ##       <int>	    <int>	<int>	<int>	<int>	<int>
    ## 1     1          2	    3	    4	    5       6
    ## 2     2          3	    4	    5	    6       7
    ##########################################################
    

    raw_data$rt <- ifelse(raw_data$trialtype == "T", 1000,
                           ifelse(raw_data$trialtype == "FS", 0,
                                  ifelse(raw_data$rt >= 500, 750, raw_data$rt)))

    # Specify phases
    raw_data$phase <-  ifelse(study=="OPSI", ifelse(as.integer(raw_data$day) < 15, "Baseline",
                                                    ifelse(as.integer(raw_data$day) < 43, "Stress", "Recovery")),
                              ifelse(study=="OPSII", ifelse(as.integer(raw_data$day) < 8, "Baseline",
                                                            ifelse(as.integer(raw_data$day) < 28, "Stress", "Recovery")),
                                     ifelse(study=="SERE", ifelse(as.integer(raw_data$day) < 3,"Baseline",
                                                                  ifelse(as.integer(raw_data$day) < 20,"Stress","Recovery")),
                                            ifelse(study=="HIGH", ifelse(as.integer(raw_data$day) < 20,"Baseline","Stress"), "UNKNOWN"))))
    
    raw_data$phase <- factor(raw_data$phase, c("Baseline","Stress","Recovery"))

    if (study=="HIGH") {
        raw_data$study_event_id <- ifelse(as.integer(raw_data$day) < 20, "B1",
                                          ifelse(as.integer(raw_data$day) < 30, "A1",
                                                 ifelse(as.integer(raw_data$day) < 40, "A2", "A3")))
        }
    
    #raw_data$day <- ifelse(study=="OPSI", factor(raw_data$day,levels=c("Day 0","Day 2","Day 4","Day 5","Day 6","Day 8","Day 9","Day 10","Day 13","Day 15","Day 20","Day 22","Day 27","Day 29","Day 34","Day 36","Day 41","Day 54","EOS")),
    #                       ifelse(study=="OPSII", factor(raw_data$day, levels=c("2","4","6","12","14","17","22","27","32","38")),
    #                              ifelse(study=="SERE", factor(raw_data$day,levels=c("bt","pt","rt")),
    #                                     ifelse(study=="HIGH", factor(raw_data$day,levels=c("B1","A1","A2","A3")), "UNKNOWN"))))

    #raw_data$day <- factor(raw_data$day, levels=c("6","22","38"))
    #return(processed$freq)
    return(raw_data)
}

#########################   END CREATE MODEL FUNCTION   ############################

In [ ]:
##############################  BEGIN KLD FUNCTION   ###############################
x_grid <- seq(0, 1000, length.out = 101)

# Function to compute KDE for a given RT vector
get_kde <- function(rt_values) {
  kde <- density(rt_values, bw = "nrd0", from = 0, to = 1000)
  approx(kde$x, kde$y, xout = x_grid)$y
}

# Function to compute KL divergence for one subject
run_sa_for_subject <- function(studyid, pid, port) {
    cat(sprintf("# study: '%s' '%s'   subject: '%s'   port: '%s'\n", studyid, studyid=="OPSII", pid, port))
  dat <- switch(studyid,
    "OPSI"  = ops1_data_cleaned,
    "OPSII" = ops2_data_cleaned,
    "SERE"  = sere_data_cleaned,
    "HIGH"  = high_data_cleaned,
    ops2_data_cleaned # default OPSII
  )
    
  days_map <- switch(studyid,
    "OPSI"  = c("Day 13","Day 41","EOS"),
    "OPSII" = c("6","22","38"),
    "SERE"  = c("bt","pt","rt"),
    "HIGH"  = c("B1","A1","A2","A3"),
    c("Day 13","Day 41","EOS") # default OPSII
  )

  #print(days_map)
    
  human_dat <- dat %>% filter(subjectid == pid, study_event_id %in% days_map) #%>% group_by(subjectid, study_event_id)
  #print(head(human_dat))
  #print(unique(human_dat$phase))

  if (studyid=="HIGH") {
      human_kde <- human_dat %>%
        group_by(phase,study_event_id) %>%
        summarise(density = list(get_kde(rt)), .groups = "drop") %>%
        unnest_longer(density) %>%
        mutate(x = rep(x_grid, times = n_distinct(study_event_id))) %>%
        group_by(phase,study_event_id) %>%
        mutate(density = (density * diff(x_grid)[1]) / sum(density * diff(x_grid)[1])) %>%
        ungroup()
      } else {
      human_kde <- human_dat %>%
        group_by(phase) %>%
        summarise(density = list(get_kde(rt)), .groups = "drop") %>%
        unnest_longer(density) %>%
        mutate(x = rep(x_grid, times = n_distinct(phase))) %>%
        group_by(phase) %>%
        mutate(density = (density * diff(x_grid)[1]) / sum(density * diff(x_grid)[1])) %>%
        ungroup()
      }
  #print(human_kde)
  #print(subset(human_kde,phase=="Stress"))
    
  full_kl_data <- tibble(study=character(),
                         subjectid=character(),
                         sa_run=integer(),
                         phase=character(),
                         study_event_id = character(),
                         bout=integer(),
                         kl_val=double(), 
                         params=list())
  current_phase <- "Baseline"
  cur_day <- days_map[1]
  boutid <- 0
  run_num <- 0
    #Get sleep schedule and bouts
    if (studyid=="OPSI") {
        start_times <- ops1_data_cleaned %>% select(subjectid, study_event_id, days_fix,StartDT) %>% filter(!is.na(StartDT))
        n_days <- as.integer(subset(start_times, subjectid==pid & study_event_id == "EOS")$days_fix)
        sleep_sched <- vector("list", n_days+1)
        for (i in seq_len(n_days+1)) {
            sleep_sched[[i]] <- list(((i-1)*24)+6,((i-1)*24)+23)
        }
        bouts <- subset(start_times, subjectid==pid) %>% mutate(hour = as.integer(format(round(StartDT, units="hours"), format="%H")),
                                                                bouts = (as.integer(days_fix)*24)+hour)
        bouts <- bouts$bouts
    } else if (studyid=="OPSII") {
        start_times <- ops2_data_cleaned %>% select(subjectid, study_event_id, days, days_fix,StartDT) %>% filter(!is.na(StartDT))
        n_days <- as.integer(subset(start_times,subjectid==pid & study_event_id == "38")$days_fix)
        sleep_sched <- vector("list", n_days)
        skips <- 0
        if (pid %in% c("A00-07-2030","A00-07-4258")) { 
            for (i in 1:4) {
                wake <- ((i-1)*24)+6
                sleep <- ((i-1)*24)+23
                sleep_sched[[i]] <- c(wake, sleep)
                skips <- skips+1
                }}
        for (i in seq_len(n_days)) {
            wake <- ifelse(length(subset(ops2_sleep_data, subjectid == pid & days == i-1)$Wake__local_)==0,
                           ((i+skips-1)*24)+6,
                           ifelse(is.na(subset(ops2_sleep_data, subjectid == pid & days == i-1)$Wake__local_),
                                  ifelse(i %in% c(11,12,13,16,17,18,21,22,23,26,27,28),
                                         ((i+skips-1)*24)+5,
                                         ((i+skips-1)*24)+6),
                                  ((i+skips-1)*24)+as.integer(format(round(subset(ops2_sleep_data, subjectid == pid & days == i-1)$Wake__local_, units="hours"), format="%H"))))
            sleep <- ifelse(is.na(subset(ops2_sleep_data, subjectid == pid & days == i)$Sleep_onset__local_),
                            ifelse(i %in% c(10,11,12,15,16,17,20,21,22,25,26,27),
                                   ((i+skips-1)*24)+25,
                                   ifelse(i %in% c(13,18,23,28),
                                          ((i+skips-1)*24)+22,
                                          ((i+skips-1)*24)+23)),
                            ((i+skips-1)*24)+as.integer(round(difftime(subset(ops2_sleep_data, subjectid == pid & days == i)$Sleep_onset__local_, 
                                                                 as.POSIXct(subset(ops2_sleep_data, subjectid == pid & days == i)$date), 
                                                                 units = "hours"))))
            sleep_sched[[i+skips]] <- c(wake, sleep)
        }
        bouts <- subset(start_times, subjectid==pid) %>% mutate(hour = as.integer(format(round(StartDT, units="hours"), format="%H")),
                                                                bouts = ((as.integer(days)-1)*24)+hour)
        bouts <- bouts$bouts

    } else if (studyid=="SERE") {
        sleep_sched <- vector("list", 45)
        for (i in seq_len(45)) {
            wake <- ifelse(i %in% c(11,12,13,14,15,16,17,18),
                           ((i-1)*24)+4,
                           ((i-1)*24)+5)
            sleep <- ifelse(i %in% c(11,12,13,14,15,16,17),
                            ((i-1)*24)+25,
                            ((i-1)*24)+22)
            sleep_sched[[i]] <- c(wake, sleep)
        }
        bouts <- c(37,421,1069)
    } else if (studyid=="HIGH") {
        n_days <- 44
        sleep_sched <- vector("list", n_days)
        for (i in seq_len(n_days)) {
            wake <- ifelse(length(subset(high_sleep_data, subjectid == pid & days == i-1)$wake_onset)==0,
                           ((i-1)*24)+6,
                           ifelse(is.na(subset(high_sleep_data, subjectid == pid & days == i-1)$wake_onset),
                                  ((i-1)*24)+6,
                                  ((i-1)*24)+as.integer(format(round(subset(high_sleep_data, subjectid == pid & days == i-1)$wake_onset, units="hours"), format="%H"))))
            sleep <- ifelse(is.na(subset(high_sleep_data, subjectid == pid & days == i)$sleep_onset),
                            ((i-1)*24)+23,
                            ((i-1)*24)+as.integer(round(difftime(subset(high_sleep_data, subjectid == pid & days == i)$sleep_onset, 
                                                                 as.POSIXct(subset(high_sleep_data, subjectid == pid & days == i)$date), 
                                                                 units = "hours"))))
            sleep_sched[[i]] <- c(wake, sleep)
        }
        bouts <- c(181,541,781,1045)
    } else {
        sleep_sched <- studyid
        bouts <- ifelse(studyid=="OPSI",ops1_bouts,
                        ifelse(studyid=="OPSII",ops2_bouts,
                               ifelse(studyid=="SERE",sere_bouts,
                                      ifelse(studyid=="HIGH",high_bouts,ops2_bouts))))
    }

    #print(sleep_sched)
    #print(bouts)
    
  kl_fun <- function(params) {
    model_data <- run_model(study = studyid, sched = sleep_sched, bout = boutid, params, iterations = iterations, PORT = port)
    #print(model_data)

    if (studyid=="HIGH") {
        model_kde <- model_data %>%
          group_by(phase,study_event_id) %>%
          summarise(density = list(get_kde(rt)), .groups = "drop") %>%
          unnest_longer(density) %>%
          mutate(x = rep(x_grid, times = n_distinct(study_event_id))) %>%
          group_by(phase,study_event_id) %>%
          mutate(density = (density * diff(x_grid)[1]) / sum(density * diff(x_grid)[1])) %>%
          ungroup()
        } else {
        model_kde <- model_data %>%
          group_by(phase) %>%
          summarise(density = list(get_kde(rt)), .groups = "drop") %>%
          unnest_longer(density) %>%
          mutate(x = rep(x_grid, times = n_distinct(phase))) %>%
          group_by(phase) %>%
          mutate(density = (density * diff(x_grid)[1]) / sum(density * diff(x_grid)[1])) %>%
          ungroup()
        }
    #print(model_kde)
    
    current_phase <<- paste(unique(model_kde$phase))
    #print(current_phase)
    #print(cur_day)
    #print(subset(human_kde,phase==current_phase))  
    if (studyid=="HIGH") {
        x <- merge(subset(human_kde,study_event_id==cur_day), model_kde, by = c("phase", "x"))
        } else {
        x <- merge(subset(human_kde,phase==current_phase), model_kde, by = c("phase", "x"))
        }
    x <- x[order(x$phase, x$x), ]
    #print(x)
      
    KL_data <- x %>%
      group_by(phase) %>%
      summarise(KL = suppressMessages(KL(rbind(density.x, density.y), unit = 'log')))
    #print(KL_data)
    #KL_mean <- mean(KL_data$KL, na.rm = TRUE)
      
    full_kl_data <<- add_row(full_kl_data, study = studyid, subjectid = pid, sa_run = run_num, phase = current_phase, study_event_id = cur_day, bout = boutid, kl_val = KL_data$KL, params = list(round(params,2)))
    #print(full_kl_data)
      
    return(KL_data$KL)
  }

    ###RUN LOOP TO FIND BEST PARAMS, IN LOOP OVER BOUTS
    ###RESULT IS A LIST OF OPTIM_SA_RESULTS AND FULL_KL_DATA FOR EACH BOUT FOR THE PARTICIPANT
  optim_sa_results_full <- vector("list", length(bouts))
  names(optim_sa_results_full) <- paste0("bout_", bouts)
    
  cur_params <- init_params
  dist_thresh <- 0

  for (i in seq_along(bouts)) {
      boutid <- bouts[i]
      cur_day <- days_map[i]
      #print(boutid)
      #print(cur_params)
      prior_optim_sa_results <- NULL
      optim_kl_val <- NULL
      run_num <- 1
      # loop until SA method doesn't find a better set of parameters 
      repeat {
          # Run Simulated Annealing Optimization
          optim_sa_results <- optim_sa(
              fun = kl_fun,
              start = cur_params,
              maximization = FALSE,
              trace = TRUE,
              lower = par_lo,
              upper = par_hi,
              control = list(t0 = t0, nlimit = nlimit)
          )

          cur_params <- round(optim_sa_results$par,2)
          cur_params <- pmax(par_lo, pmin(cur_params, par_hi))
          optim_kl_val <- optim_sa_results$function_value
          #print(cur_params)
          if (!is.null(prior_optim_sa_results)) {
              d <- as.numeric(dist(rbind(round(prior_optim_sa_results$par,2),cur_params)))
              if (d <= dist_thresh | optim_kl_val >= prior_optim_sa_results$function_value) {
                  break
              }
          }
          run_num <- run_num+1
          prior_optim_sa_results <- optim_sa_results
      }
      ###save prior to last results if current KL val is higher than prior
      if (optim_kl_val >= prior_optim_sa_results$function_value) {
          cur_params <- round(prior_optim_sa_results$par,2)
          cur_params <- pmax(par_lo, pmin(cur_params, par_hi))
          prior_optim_sa_results$study <- studyid
          prior_optim_sa_results$phase <- current_phase
          prior_optim_sa_results$study_event_id <- cur_day
          prior_optim_sa_results$bout <- boutid
          optim_sa_results_full[[i]] <- prior_optim_sa_results
      } else {
          cur_params <- round(optim_sa_results$par,2)
          cur_params <- pmax(par_lo, pmin(cur_params, par_hi))
          optim_sa_results$study <- studyid
          optim_sa_results$phase <- current_phase
          optim_sa_results$study_event_id <- cur_day
          optim_sa_results$bout <- boutid
          optim_sa_results_full[[i]] <- optim_sa_results
      }
  }


    # Shut down lisp process for subject
    # open socket
    sock <- make.socket(host = HOST, port = port, server = FALSE)
    # connect to server and send exit signal
    write.socket(sock, paste(toJSON(list(exit="exit"), auto_unbox = TRUE),"\r\n"))
    cat(sprintf("# closing lisp at port '%s'\n", port))
    # close socket
    on.exit(close.socket(sock))
    #write data out to file
    return(list(optim_sa_results = optim_sa_results_full,kl_summary = full_kl_data))
}

###############################  END KLD FUNCTION   ################################

# OPSII Parameter Optimization

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSII  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(ops2_data_cleaned$subjectid)[1:10] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSII",length(participants))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS2_OptimSA_results_list_1) <- participants

OPS2_OptimSA_results_1 <- map(OPS2_OptimSA_results_list_1, "optim_sa_results")
OPS2_KL_summaries_1 <- map_dfr(OPS2_OptimSA_results_list_1, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSII_KLresults_p1-10.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS2_OptimSA_results_1)
colnames(OPS2_OptimSA_results_1[[1]][[1]][[3]])

trace <- imap_dfr(OPS2_OptimSA_results_1, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS2_OptimSA_results_1[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSII_OptimSAtrace_p1-10.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS2_KL_summaries_1 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSII_fullKLdata_p1-10.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING   #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

######################################### PARTICIPANTS 11-20 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants_2 <- unique(ops2_data_cleaned$subjectid)[11:20] #unique(ops2_data_cleaned$subjectid)[1:2]
participants_2

# Assign unique ports
base_port_2 <- 9850
ports_2 <- base_port_2 + seq_along(participants_2) - 1
ports_2

study_list <- rep("OPSII",length(participants_2))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants_2, port = ports_2), run_sa_for_subject)

names(OPS2_OptimSA_results_list_2) <- participants_2

OPS2_OptimSA_results_2 <- map(OPS2_OptimSA_results_list_2, "optim_sa_results")
OPS2_KL_summaries_2 <- map_dfr(OPS2_OptimSA_results_list_2, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSII_KLresults_p11-20.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS2_OptimSA_results_2)
colnames(OPS2_OptimSA_results_2[[1]][[1]][[3]])

trace <- imap_dfr(OPS2_OptimSA_results_2, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS2_OptimSA_results_2[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSII_OptimSAtrace_p11-20.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS2_KL_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSII_fullKLdata_p11-20.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING   #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

######################################### PARTICIPANTS 11-20 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants_3 <- unique(ops2_data_cleaned$subjectid)[21:30] #unique(ops2_data_cleaned$subjectid)[1:2]
participants_3

# Assign unique ports
base_port_3 <- 9850
ports_3 <- base_port_3 + seq_along(participants_3) - 1
ports_3

study_list <- rep("OPSII",length(participants_3))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_3 <- future_pmap(list(studyid = study_list, pid = participants_3, port = ports_3), run_sa_for_subject)

names(OPS2_OptimSA_results_list_3) <- participants_3

OPS2_OptimSA_results_3 <- map(OPS2_OptimSA_results_list_3, "optim_sa_results")
OPS2_KL_summaries_3 <- map_dfr(OPS2_OptimSA_results_list_3, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSII_KLresults_p21-30.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS2_OptimSA_results_3)
colnames(OPS2_OptimSA_results_3[[1]][[1]][[3]])

trace <- imap_dfr(OPS2_OptimSA_results_3, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS2_OptimSA_results_3[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSII_OptimSAtrace_p21-30.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS2_KL_summaries_3 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSII_fullKLdata_p21-30.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# OPSI Parameter Optimization

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(ops1_data_cleaned$subjectid)[1:12] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_1) <- participants

OPS1_OptimSA_results_1 <- map(OPS1_OptimSA_results_list_1, "optim_sa_results")
OPS1_KL_summaries_1 <- map_dfr(OPS1_OptimSA_results_list_1, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSI_KLresults_p1-12.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS1_OptimSA_results_1)
colnames(OPS1_OptimSA_results_1[[1]][[1]][[3]])

trace <- imap_dfr(OPS1_OptimSA_results_1, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS1_OptimSA_results_1[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSI_OptimSAtrace_p1-12.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS1_KL_summaries_1 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSI_fullKLdata_p1-12.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops1_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(ops1_data_cleaned$subjectid)[13:24] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_2) <- participants

OPS1_OptimSA_results_2 <- map(OPS1_OptimSA_results_list_2, "optim_sa_results")
OPS1_KL_summaries_2 <- map_dfr(OPS1_OptimSA_results_list_2, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSI_KLresults_p13-24.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS1_OptimSA_results_2)
colnames(OPS1_OptimSA_results_2[[1]][[1]][[3]])

trace <- imap_dfr(OPS1_OptimSA_results_2, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS1_OptimSA_results_2[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSI_OptimSAtrace_p13-24.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS1_KL_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSI_fullKLdata_p13-24.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(ops1_data_cleaned$subjectid)[25:36] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_3 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_3) <- participants

OPS1_OptimSA_results_3 <- map(OPS1_OptimSA_results_list_3, "optim_sa_results")
OPS1_KL_summaries_3 <- map_dfr(OPS1_OptimSA_results_list_3, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSI_KLresults_p25-36.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS1_OptimSA_results_3)
colnames(OPS1_OptimSA_results_3[[1]][[1]][[3]])

trace <- imap_dfr(OPS1_OptimSA_results_3, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS1_OptimSA_results_3[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSI_OptimSAtrace_p25-36.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS1_KL_summaries_3 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSI_fullKLdata_p25-36.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(ops1_data_cleaned$subjectid)[37:48] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_4 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_4) <- participants

OPS1_OptimSA_results_4 <- map(OPS1_OptimSA_results_list_4, "optim_sa_results")
OPS1_KL_summaries_4 <- map_dfr(OPS1_OptimSA_results_list_4, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_4, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_OPSI_KLresults_p37-48.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(OPS1_OptimSA_results_4)
colnames(OPS1_OptimSA_results_4[[1]][[1]][[3]])

trace <- imap_dfr(OPS1_OptimSA_results_4, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(OPS1_OptimSA_results_4[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_OPSI_OptimSAtrace_p37-48.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- OPS1_KL_summaries_4 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_OPSI_fullKLdata_p37-48.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# SERE Parameter Optimization

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[1:9] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_1) <- participants

SERE_OptimSA_results_1 <- map(SERE_OptimSA_results_list_1, "optim_sa_results")
SERE_KL_summaries_1 <- map_dfr(SERE_OptimSA_results_list_1, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p1-10.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_1)
colnames(SERE_OptimSA_results_1[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_1, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_1[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p1-10.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- SERE_KL_summaries_1 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p1-10.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[10:18] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9851
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_2) <- participants

SERE_OptimSA_results_2 <- map(SERE_OptimSA_results_list_2, "optim_sa_results")
SERE_KL_summaries_2 <- map_dfr(SERE_OptimSA_results_list_2, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p10-18.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_2)
colnames(SERE_OptimSA_results_2[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_2, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_2[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p10-18.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- SERE_KL_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p10-18.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[19:27] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_3 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_3) <- participants

SERE_OptimSA_results_3 <- map(SERE_OptimSA_results_list_3, "optim_sa_results")
SERE_KL_summaries_3 <- map_dfr(SERE_OptimSA_results_list_3, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p19-27.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_3)
colnames(SERE_OptimSA_results_3[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_3, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_3[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p19-27.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- SERE_KL_summaries_3 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p19-27.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[28:36] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_4 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_4) <- participants

SERE_OptimSA_results_4 <- map(SERE_OptimSA_results_list_4, "optim_sa_results")
SERE_KL_summaries_4 <- map_dfr(SERE_OptimSA_results_list_4, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_4, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p28-36.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_4)
colnames(SERE_OptimSA_results_4[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_4, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_4[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p28-36.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- SERE_KL_summaries_4 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p28-36.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[37:45] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_5 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_5) <- participants

SERE_OptimSA_results_5 <- map(SERE_OptimSA_results_list_5, "optim_sa_results")
SERE_KL_summaries_5 <- map_dfr(SERE_OptimSA_results_list_5, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_5, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p37-45.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_5)
colnames(SERE_OptimSA_results_5[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_5, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_5[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p37-45.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- SERE_KL_summaries_5 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p37-45.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[46:54] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_6 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_6) <- participants

SERE_OptimSA_results_6 <- map(SERE_OptimSA_results_list_6, "optim_sa_results")
SERE_KL_summaries_6 <- map_dfr(SERE_OptimSA_results_list_6, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_6, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p46-54.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_6)
colnames(SERE_OptimSA_results_6[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_6, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_6[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p46-54.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- SERE_KL_summaries_6 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p46-54.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(sere_data_cleaned$subjectid)[55:63] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("SERE",length(participants))

# Run in parallel across all subjects
SERE_OptimSA_results_list_7 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(SERE_OptimSA_results_list_7) <- participants

SERE_OptimSA_results_7 <- map(SERE_OptimSA_results_list_7, "optim_sa_results")
SERE_KL_summaries_7 <- map_dfr(SERE_OptimSA_results_list_7, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(SERE_OptimSA_results_7, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_SERE_KLresults_p55-63.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(SERE_OptimSA_results_7)
colnames(SERE_OptimSA_results_7[[1]][[1]][[3]])

trace <- imap_dfr(SERE_OptimSA_results_7, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(SERE_OptimSA_results_7[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_SERE_OptimSAtrace_p55-63.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_SERE_fullKLdata_p55-63.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# HIGH Parameter Optimization

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  HIGH  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(high_data_cleaned$subjectid)[1:8] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("HIGH",length(participants))

# Run in parallel across all subjects
HIGH_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(HIGH_OptimSA_results_list_1) <- participants

HIGH_OptimSA_results_1 <- map(HIGH_OptimSA_results_list_1, "optim_sa_results")
HIGH_KL_summaries_1 <- map_dfr(HIGH_OptimSA_results_list_1, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(HIGH_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_HIGH_KLresults_p1-8.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(HIGH_OptimSA_results_1)
colnames(HIGH_OptimSA_results_1[[1]][[1]][[3]])

trace <- imap_dfr(HIGH_OptimSA_results_1, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(HIGH_OptimSA_results_1[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_HIGH_OptimSAtrace_p1-8.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_HIGH_fullKLdata_p1-8.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  HIGH  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- sere_bouts

## Create lower and upper bounds of parameters
par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40


######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(high_data_cleaned$subjectid)[9:15] #unique(ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("HIGH",length(participants))

# Run in parallel across all subjects
HIGH_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(HIGH_OptimSA_results_list_2) <- participants

HIGH_OptimSA_results_2 <- map(HIGH_OptimSA_results_list_2, "optim_sa_results")
HIGH_KL_summaries_2 <- map_dfr(HIGH_OptimSA_results_list_2, "kl_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(HIGH_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(round(res$par, 2))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9)
head(df_wide)

In [ ]:
filename <- "PVT_KL_Individual_HIGH_KLresults_p9-15.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
names(HIGH_OptimSA_results_2)
colnames(HIGH_OptimSA_results_2[[1]][[1]][[3]])

trace <- imap_dfr(HIGH_OptimSA_results_2, function(phase_list, subject) {
    imap_dfr(phase_list, function(res, phase) {
        tibble(
            study     = res$study,
            subjectid = subject,
            phase     = res$phase,
            study_event_id = res$study_event_id,
            bout      = res$bout,
            trace     = res$trace
    )
  })
})
head(trace)

trace_wide <- trace %>% mutate(trace = as.data.frame(trace)) %>% unnest_wider(trace, names_sep = "_")
names(trace_wide) <- c("study","subjectid","phase","study_event_id","bout",colnames(HIGH_OptimSA_results_2[[1]][[1]][[3]]))
trace_wide

In [ ]:
filename <- "PVT_KL_Individual_HIGH_OptimSAtrace_p9-15.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
KL_summaries_wide <- HIGH_KL_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9)
KL_summaries_wide

In [ ]:
head(KL_summaries_wide)
filename <- "PVT_KL_Individual_HIGH_fullKLdata_p9-15.csv"
write.csv(KL_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# Combine all datasets and save to file

In [ ]:
# Base path
path <- "C:/Users/ecranford/Desktop/IHMC/Projects/ROPER/CogModels/PVT/PVT_ParamOpt"

# List filenames with filepaths
files <- file.path(path, c(
    "PVT_KL_Individual_OPSI_KLresults_p1-12.csv",
    "PVT_KL_Individual_OPSI_KLresults_p13-24.csv",
    "PVT_KL_Individual_OPSI_KLresults_p25-36.csv",
    "PVT_KL_Individual_OPSI_KLresults_p37-48.csv",
    "PVT_KL_Individual_OPSII_KLresults_p1-10.csv",
    "PVT_KL_Individual_OPSII_KLresults_p11-20.csv",
    "PVT_KL_Individual_OPSII_KLresults_p21-30.csv",
    "PVT_KL_Individual_SERE_KLresults_p1-9.csv",
    "PVT_KL_Individual_SERE_KLresults_p10-18.csv",
    "PVT_KL_Individual_SERE_KLresults_p19-27.csv",
    "PVT_KL_Individual_SERE_KLresults_p28-36.csv",
    "PVT_KL_Individual_SERE_KLresults_p37-45.csv",
    "PVT_KL_Individual_SERE_KLresults_p46-54.csv",
    "PVT_KL_Individual_SERE_KLresults_p55-63.csv"))

# Read files
df_list <- lapply(files, read.csv)
# Combine data
combined_df <- do.call(rbind, df_list)

# Sort data by study, subjectid, phase (Stress, Baseline, Recovery)
# Ensure "phase" is an ordered factor
combined_df$phase <- factor(
  combined_df$phase,
  levels = c("Baseline", "Stress", "Recovery"),
  ordered = TRUE
)

# Order the dataframe
combined_df <- combined_df[
  order(
    combined_df$study,
    combined_df$subjectid,
    combined_df$phase
  ),
]

# Write combined dataset file
write.csv(combined_df, "PVT_KL_Individual_KLresults_FullData.csv", row.names = FALSE)
# Save to the same folder
write.csv(
  combined_df,
  file.path(path, "ROPER_CompuCognome_PVTonly_FullData.csv"),
  row.names = FALSE
)


In [ ]:
# Base path
path <- "C:/Users/ecranford/Desktop/IHMC/Projects/ROPER/CogModels/PVT/PVT_ParamOpt"

# List filenames with filepaths
files <- file.path(path, c(
    "PVT_KL_Individual_OPSI_KLresults_p1-12.csv",
    "PVT_KL_Individual_OPSI_KLresults_p13-24.csv",
    "PVT_KL_Individual_OPSI_KLresults_p25-36.csv",
    "PVT_KL_Individual_OPSI_KLresults_p37-48.csv",
    "PVT_KL_Individual_OPSII_KLresults_p1-10.csv",
    "PVT_KL_Individual_OPSII_KLresults_p11-20.csv",
    "PVT_KL_Individual_OPSII_KLresults_p21-30.csv",
    "PVT_KL_Individual_SERE_KLresults_p1-9.csv",
    "PVT_KL_Individual_SERE_KLresults_p10-18.csv",
    "PVT_KL_Individual_SERE_KLresults_p19-27.csv",
    "PVT_KL_Individual_SERE_KLresults_p28-36.csv",
    "PVT_KL_Individual_SERE_KLresults_p37-45.csv",
    "PVT_KL_Individual_SERE_KLresults_p46-54.csv",
    "PVT_KL_Individual_SERE_KLresults_p55-63.csv",
    "PVT_KL_Individual_HIGH_KLresults_p1-8.csv",
    "PVT_KL_Individual_HIGH_KLresults_p9-15.csv"))

# Read files
df_list <- lapply(files, read.csv)
# Combine data
combined_df <- do.call(rbind, df_list)

# Sort data by study, subjectid, phase (Stress, Baseline, Recovery)
# Ensure "phase" is an ordered factor
combined_df$phase <- factor(
  combined_df$phase,
  levels = c("Baseline", "Stress", "Recovery"),
  ordered = TRUE
)

combined_df$study <- factor(
  combined_df$study,
  levels = c("OPSI", "OPSII", "SERE","HIGH"),
  ordered = TRUE
)

# Order the dataframe
combined_df <- combined_df[
  order(
    combined_df$study,
    combined_df$subjectid,
    combined_df$phase, 
    combined_df$study_event_id
  ),
]

# Save to the same folder
write.csv(
  combined_df,
  file.path(path, "ROPER_CompuCognome_PVTonly_FullData_AllStudies.csv"),
  row.names = FALSE
)


# NBACK PARAMETER OPTIMISATION FUNCTIONS

In [ ]:
### Import PVT parameters data
#PVT_KL_Individual_KLresults_FullData.csv
#ROPER_CompuCognome_PVTonly_FullData_AllStudies.csv
full_pvt_params <- read.csv("ROPER_CompuCognome_PVTonly_FullData_AllStudies.csv",header=TRUE)
full_pvt_params

In [ ]:
########################   START CREATE MODEL FUNCTION   ###########################
## Create our model function
## Send message to model as json of key:value pairs
## where key = parameter name and value = parameter value
mean_pvt_params<-c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)

run_model <- function(study,sched,bout,params,iterations,PORT,fix_procedural=FALSE,pvt_params=mean_pvt_params) {
    #startTime <- Sys.time()
    #cat(sprintf("# Start Time: %s\n",startTime))
    if (fix_procedural) {
        params <- list(dat = pvt_params[1], egs = pvt_params[2], ut = pvt_params[3], iu = pvt_params[4], 
                       utmc = pvt_params[5], utbmc = pvt_params[6], fpdec = pvt_params[7], fpmc = pvt_params[8], fpbmc = pvt_params[9], 
                       fdbmc = params[1], fdc = params[2], fddec = params[3], 
                       ans = params[4], bll = params[5], mp = params[6], lf = params[7], 
                       bouts = list(bout), schedule = sched, numiterations = iterations)
        } else {
        params <- list(dat = params[1], egs = params[2], ut = params[3], iu = params[4], 
                       utmc = params[5], utbmc = params[6], fpdec = params[7], fpmc = params[8], fpbmc = params[9], 
                       fdbmc = params[10], fdc = params[11], fddec = params[12], 
                       ans = params[13], bll = params[14], mp = params[15], lf = params[16], 
                       bouts = list(bout), schedule = sched, numiterations = iterations)
        }
    cat(sprintf("# request '%s'\n", toJSON(params)))
    params <- toJSON(params, auto_unbox = TRUE)
    cat(sprintf("# sending '%s'\n", params))

    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    sock <- make.socket(host = HOST, port = PORT, server = FALSE)
    on.exit(close.socket(sock))
    
    # Connect to server and send data
    write.socket(sock, paste(params,"\r\n"))
    cat(sprintf("# connected to port '%s'\n", PORT))

    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    # Receive data from the server and shut down
    #received <- character(0)
    received <- character()
    repeat {
        ss <- read.socket(sock)
        if (ss == "") break
        received <- c(received, ss)
    }
    
    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    received <- paste(received, collapse = "")

    cat(sprintf("# received message of length %s\n", nchar(received)))
    raw_data <- fromJSON(fromJSON(received))

    #cat(sprintf("# Elapsed Time: %s\n",Sys.time() - startTime))
    
    ############## EXAMPLE received DATA FRAME ###############
    ##########################################################
    ##       iteration	day	    bout	curtime	delay	rt
    ##       <int>	    <int>	<int>	<int>	<int>	<int>
    ## 1     1          2	    3	    4	    5       6
    ## 2     2          3	    4	    5	    6       7
    ##########################################################

    # Specify phases
    raw_data$phase <-  ifelse(study=="OPSI", ifelse(as.integer(raw_data$day) < 15, "Baseline",
                                                    ifelse(as.integer(raw_data$day) < 43, "Stress", "Recovery")),
                              ifelse(study=="OPSII", ifelse(as.integer(raw_data$day) < 8, "Baseline",
                                                            ifelse(as.integer(raw_data$day) < 28, "Stress", "Recovery")),
                                     ifelse(study=="HIGH", ifelse(as.integer(raw_data$day) < 20,"Baseline","Stress"), "UNKNOWN")))
    
    raw_data$phase <- factor(raw_data$phase, c("Baseline","Stress","Recovery"))

    if (study=="HIGH") {
        raw_data$study_event_id <- ifelse(as.integer(raw_data$day) < 20, "B1",
                                          ifelse(as.integer(raw_data$day) < 30, "A1",
                                                 ifelse(as.integer(raw_data$day) < 40, "A2", "A3")))
        raw_data$study_event_id <- factor(raw_data$study_event_id, c("B1","A1","A2","A3"))
    } else if (study == "OPSI") {
        raw_data$study_event_id <- ifelse(as.integer(raw_data$day) < 15, "Day 13",
                                          ifelse(as.integer(raw_data$day) < 43, "Day 41","EOS"))
        raw_data$study_event_id <- factor(raw_data$study_event_id, c("Day 13","Day 41","EOS"))
    } else if (study=="OPSII") {
        raw_data$study_event_id <- ifelse(as.integer(raw_data$day) < 8, "6",
                                          ifelse(as.integer(raw_data$day) < 28, "22","38"))
        raw_data$study_event_id <- factor(raw_data$study_event_id, c("6","22","38"))
    }

    #return(processed$freq)
    return(raw_data)
}

#########################   END CREATE MODEL FUNCTION   ############################

In [ ]:
##############################  BEGIN KLD FUNCTION   ###############################

# Function to compute KDE for a given RT or dprime vector
get_kde <- function(model_values, human_value, metric = "dprime", eps = 1e-4) {
  if (!is.finite(human_value)) {
      #print("returning NA")
      return(NA_real_)
  }
  model_values <- model_values[is.finite(model_values)]
  if (length(model_values) < 2 || sd(model_values) == 0) {
    return(eps)
  }

  if (metric == "dprime") {
    from_val <- min(model_values, -5, na.rm = TRUE)
    to_val   <- max(model_values,  5, na.rm = TRUE)
  } else if (metric == "mrt") {
    from_val <- min(model_values, 0, na.rm = TRUE)
    to_val   <- max(model_values, 3, na.rm = TRUE)
  } else {
    from_val <- min(model_values, na.rm = TRUE)
    to_val   <- max(model_values, na.rm = TRUE)
  }
    
  kde <- density(model_values, bw = "nrd0", from = from_val, to = to_val)
  y <- approx(kde$x, kde$y, xout = human_value, rule = 2)$y
  if (!is.finite(y) || is.na(y)) {
    return(eps)
  }
  max(y, eps)
}

# Function to compute MLE for one subject
run_sa_for_subject <- function(studyid, pid, port) {
    cat(sprintf("# study: '%s' '%s'   subject: '%s'   port: '%s'\n", studyid, studyid=="OPSII", pid, port))
  dat <- switch(studyid,
    "OPSI"  = nback_ops1_data_cleaned,
    "OPSII" = nback_ops2_data_cleaned,
    "SERE"  = nback_sere_data_cleaned,
    "HIGH"  = nback_high_data_cleaned,
    nback_ops2_data_cleaned # default OPSII
  )
    
  days_map <- switch(studyid,
    "OPSI"  = c("Day 13","Day 41","EOS"),
    "OPSII" = c("6","22","38"),
    "SERE"  = c("bt","pt","rt"),
    "HIGH"  = c("B1","A1","A2","A3"),
    c("6","22","38") # default OPSII
  )

  ind_pvt_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02) #default pvt params
  fix_proc <- fix_procedural
  #print(days_map)
    
  human_dat <- dat %>% filter(subjectid == pid, study_event_id %in% days_map) %>% 
    filter(!(TrialNo=="NA")) %>% group_by(phase,study_event_id, SessBlockID) %>%
    mutate(hit = ifelse(TrialType == "target" & Response == 1, 1, 0),
           miss = ifelse(TrialType == "target" & Response == 0, 1, 0),
           fa = ifelse(TrialType == "empty" & Response == -1, 1, 0),
           cr = ifelse(TrialType == "empty" & Response == 1, 1, 0)) %>%
    summarise(mrt = mean(ResponseTime[hit==1 | fa==1]),
              hr_adj = (sum(hit)+0.5)/(sum(hit | miss)+1.0),
              far_adj = (sum(fa)+0.5)/(sum(fa | cr)+1.0)
             ) %>% 
    #ungroup() %>%
    mutate(zhr = replace_na(qnorm(hr_adj),0),
           zfar = replace_na(qnorm(far_adj),0),
           dprime = zhr - zfar)
  #print(head(human_dat))
  #print(unique(human_dat$phase))
    
  full_mle_data <- tibble(study=character(),
                         subjectid=character(),
                         sa_run=integer(),
                         phase=character(),
                         study_event_id = character(),
                         bout=integer(),
                         mle_val=double(), 
                         params=list())
  current_phase <- "Baseline"
  cur_day <- days_map[1]
  boutid <- 0
  run_num <- 0
    #Get sleep schedule and bouts
    if (studyid=="OPSI") {
        start_times <- nback_ops1_data_cleaned %>% select(subjectid, study_event_id, days_fix,StartDT,SessBlockID) %>% filter(!is.na(StartDT),SessBlockID==1)
        n_days <- as.integer(subset(start_times, subjectid==pid & study_event_id == "EOS")$days_fix)
        sleep_sched <- vector("list", n_days+1)
        for (i in seq_len(n_days+1)) {
            sleep_sched[[i]] <- list(((i-1)*24)+6,((i-1)*24)+23)
        }
        bouts <- subset(start_times, subjectid==pid) %>% mutate(hour = as.integer(format(round(StartDT, units="hours"), format="%H")),
                                                                bouts = (as.integer(days_fix)*24)+hour)
        bouts <- bouts$bouts
    } else if (studyid=="OPSII") {
        start_times <- nback_ops2_data_cleaned %>% select(subjectid, study_event_id, days, days_fix,StartDT,SessBlockID) %>% filter(!is.na(StartDT),SessBlockID==1)
        n_days <- as.integer(subset(start_times,subjectid==pid & study_event_id == "38")$days_fix)
        sleep_sched <- vector("list", n_days)
        skips <- 0
        if (pid %in% c("ROP078","ROP084")) { 
            for (i in 1:4) {
                wake <- ((i-1)*24)+6
                sleep <- ((i-1)*24)+23
                sleep_sched[[i]] <- c(wake, sleep)
                skips <- skips+1
                }}
        for (i in seq_len(n_days)) {
            wake <- ifelse(length(subset(ops2_sleep_data, subjectid == pid & days == i-1)$Wake__local_)==0,
                           ((i+skips-1)*24)+6,
                           ifelse(is.na(subset(ops2_sleep_data, subjectid == pid & days == i-1)$Wake__local_),
                                  ifelse(i %in% c(11,12,13,16,17,18,21,22,23,26,27,28),
                                         ((i+skips-1)*24)+5,
                                         ((i+skips-1)*24)+6),
                                  ((i+skips-1)*24)+as.integer(format(round(subset(ops2_sleep_data, subjectid == pid & days == i-1)$Wake__local_, units="hours"), format="%H"))))
            sleep <- ifelse(is.na(subset(ops2_sleep_data, subjectid == pid & days == i)$Sleep_onset__local_),
                            ifelse(i %in% c(10,11,12,15,16,17,20,21,22,25,26,27),
                                   ((i+skips-1)*24)+25,
                                   ifelse(i %in% c(13,18,23,28),
                                          ((i+skips-1)*24)+22,
                                          ((i+skips-1)*24)+23)),
                            ((i+skips-1)*24)+as.integer(round(difftime(subset(ops2_sleep_data, subjectid == pid & days == i)$Sleep_onset__local_, 
                                                                 as.POSIXct(subset(ops2_sleep_data, subjectid == pid & days == i)$date), 
                                                                 units = "hours"))))
            sleep_sched[[i+skips]] <- c(wake, sleep)
        }
        bouts <- subset(start_times, subjectid==pid) %>% mutate(hour = as.integer(format(round(StartDT, units="hours"), format="%H")),
                                                                bouts = ((as.integer(days)-1)*24)+hour)
        bouts <- bouts$bouts

    } else if (studyid=="SERE") {
        sleep_sched <- vector("list", 45)
        for (i in seq_len(45)) {
            wake <- ifelse(i %in% c(11,12,13,14,15,16,17,18),
                           ((i-1)*24)+4,
                           ((i-1)*24)+5)
            sleep <- ifelse(i %in% c(11,12,13,14,15,16,17),
                            ((i-1)*24)+25,
                            ((i-1)*24)+22)
            sleep_sched[[i]] <- c(wake, sleep)
        }
        bouts <- c(37,421,1069)
    } else if (studyid=="HIGH") {
        n_days <- 44
        sleep_sched <- vector("list", n_days)
        for (i in seq_len(n_days)) {
            wake <- ifelse(length(subset(high_sleep_data, subjectid == pid & days == i-1)$wake_onset)==0,
                           ((i-1)*24)+6,
                           ifelse(is.na(subset(high_sleep_data, subjectid == pid & days == i-1)$wake_onset),
                                  ((i-1)*24)+6,
                                  ((i-1)*24)+as.integer(format(round(subset(high_sleep_data, subjectid == pid & days == i-1)$wake_onset, units="hours"), format="%H"))))
            sleep <- ifelse(is.na(subset(high_sleep_data, subjectid == pid & days == i)$sleep_onset),
                            ((i-1)*24)+23,
                            ((i-1)*24)+as.integer(round(difftime(subset(high_sleep_data, subjectid == pid & days == i)$sleep_onset, 
                                                                 as.POSIXct(subset(high_sleep_data, subjectid == pid & days == i)$date), 
                                                                 units = "hours"))))
            sleep_sched[[i]] <- c(wake, sleep)
        }
        bouts <- c(181,541,781,1045)
    } else {
        sleep_sched <- studyid
        bouts <- ifelse(studyid=="OPSI",ops1_bouts,
                        ifelse(studyid=="OPSII",ops2_bouts,
                               ifelse(studyid=="SERE",sere_bouts,
                                      ifelse(studyid=="HIGH",high_bouts,ops2_bouts))))
    }

    #print(sleep_sched)
    #print(bouts)
    
  mle_fun <- function(params) {
    model_data <- run_model(study = studyid, sched = sleep_sched, bout = boutid, params, iterations = iterations, PORT = port, fix_procedural = fix_proc, pvt_params = ind_pvt_params)
    #print(model_data)

    model_data <- model_data %>% 
        mutate(hit = ifelse(respType == 1 & resp == 1, 1, 0),
               miss = ifelse(respType == 1 & resp == 0, 1, 0),
               fa = ifelse(respType == 0 & resp == 1, 1, 0),
               cr = ifelse(respType == 0 & resp == 0, 1, 0)) %>%
        group_by(iteration,phase,study_event_id,nback) %>%
        summarise(mrt = mean(as.numeric(respTime[respType==1&resp==1])),
                  hr_adj = (sum(hit)+0.5)/(sum(hit | miss)+1.0),
                  far_adj = (sum(fa)+0.5)/(sum(fa | cr)+1.0)) %>% 
            ungroup() %>%
        mutate(zhr = replace_na(qnorm(hr_adj),0),
               zfar = replace_na(qnorm(far_adj),0),
               dprime = zhr - zfar)
    model_data$nback <- factor(model_data$nback, levels = c(1,2))

    current_phase <<- paste(unique(model_data$phase))
    #print(current_phase)
    #print(cur_day)
    #print(subset(human_kde,phase==current_phase))  

    mle_data <- model_data %>% group_by(iteration,phase,study_event_id,nback) %>%
        rename(SessBlockID = nback) %>%
        left_join(subset(human_dat,study_event_id==cur_day), by = c("phase","study_event_id","SessBlockID")) %>%
        group_by(phase,study_event_id,SessBlockID) %>%
        summarise(ll_mrt = {dens <- get_kde(mrt.x, mrt.y[1], metric = "mrt")
                            if (is.na(dens)) 0 else log(dens)
                                },
                  ll_dprime = {dens <- get_kde(dprime.x, dprime.y[1], metric = "dprime")
                               if (is.na(dens)) 0 else log(dens)
                                   },
                  ll = ll_mrt + ll_dprime,
                  .groups = "drop") %>%
        #summarise(ll_mrt = log(get_kde(mrt.x, mrt.y[1]) + 1e-4),
        #          ll_dprime = log(get_kde(dprime.x, dprime.y[1]) + 1e-4),
        #          ll = ll_mrt + ll_dprime,
        #          .groups = "drop") %>% #kde(meanrrt.x,h=bw.nrd0(meanrrt.x),eval.points=mean(meanrrt.y))$estimate)) %>%
        ungroup()

    #print(mle_data)
    #sum(mle_data$ll)
      
    full_mle_data <<- add_row(full_mle_data, study = studyid, subjectid = pid, sa_run = run_num, phase = current_phase, study_event_id = cur_day, bout = boutid, mle_val = sum(mle_data$ll), params = list(c(ind_pvt_params, round(params,3))))
    #print(full_kl_data)
      
    return(sum(mle_data$ll))
  }

    ###RUN LOOP TO FIND BEST PARAMS, IN LOOP OVER BOUTS
    ###RESULT IS A LIST OF OPTIM_SA_RESULTS AND FULL_KL_DATA FOR EACH BOUT FOR THE PARTICIPANT
  optim_sa_results_full <- vector("list", length(bouts))
  names(optim_sa_results_full) <- paste0("bout_", bouts)
    
  cur_params <- init_params
  par_lo <- par_low
  par_hi <- par_high
  dist_thresh <- 0

  for (i in seq_along(bouts)) {
      boutid <- bouts[i]
      cur_day <- days_map[i]
      #print(boutid)
      #print(cur_params)
      prior_optim_sa_results <- NULL
      optim_mle_val <- NULL
      run_num <- 1
      # set individual pvt params
      ind_pvt_params <- full_pvt_params %>%
          filter(study == studyid, subjectid == pid, study_event_id == cur_day) %>%
          slice(1) %>% select(dat:fpbmc) %>% unlist(use.names = FALSE)

      # if participant has no pvt params data, fit across all parameters
      if (length(ind_pvt_params)==0) {
          fix_proc <- FALSE
          if (i==1) {
              cur_params <- c(0.050,0.350,2.460,2.370,-0.080,0.030,0.990,-0.010,0.030,-0.015,0.930,0.990,0.170,0.430,0.980,0.890)
              par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
              par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
              }
      }
      
      # loop until SA method doesn't find a better set of parameters 
      repeat {
          # Run Simulated Annealing Optimization
          optim_sa_results <- optim_sa(
              fun = mle_fun,
              start = cur_params,
              maximization = TRUE,
              trace = TRUE,
              lower = par_lo,
              upper = par_hi,
              control = list(t0 = t0, nlimit = nlimit)
          )

          cur_params <- round(optim_sa_results$par,3)
          cur_params <- pmax(par_lo, pmin(cur_params, par_hi))
          optim_mle_val <- optim_sa_results$function_value
          #print(cur_params)
          if (!is.null(prior_optim_sa_results)) {
              d <- as.numeric(dist(rbind(round(prior_optim_sa_results$par,3),cur_params)))
              if (d <= dist_thresh | optim_mle_val <= prior_optim_sa_results$function_value) {
                  break
              }
          }
          run_num <- run_num+1
          prior_optim_sa_results <- optim_sa_results
      }
      ###save prior to last results if current KL val is lower than prior
      if (optim_mle_val <= prior_optim_sa_results$function_value) {
          cur_params <- round(prior_optim_sa_results$par,3)
          cur_params <- pmax(par_lo, pmin(cur_params, par_hi))
          prior_optim_sa_results$study <- studyid
          prior_optim_sa_results$phase <- current_phase
          prior_optim_sa_results$study_event_id <- cur_day
          prior_optim_sa_results$bout <- boutid
          prior_optim_sa_results$pvt_params <- ind_pvt_params
          optim_sa_results_full[[i]] <- prior_optim_sa_results
      } else {
          cur_params <- round(optim_sa_results$par,3)
          cur_params <- pmax(par_lo, pmin(cur_params, par_hi))
          optim_sa_results$study <- studyid
          optim_sa_results$phase <- current_phase
          optim_sa_results$study_event_id <- cur_day
          optim_sa_results$bout <- boutid
          optim_sa_results$pvt_params <- ind_pvt_params
          optim_sa_results_full[[i]] <- optim_sa_results
      }
  }


    # Shut down lisp process for subject
    # open socket
    sock <- make.socket(host = HOST, port = port, server = FALSE)
    # connect to server and send exit signal
    write.socket(sock, paste(toJSON(list(exit="exit"), auto_unbox = TRUE),"\r\n"))
    cat(sprintf("# closing lisp at port '%s'\n", port))
    # close socket
    on.exit(close.socket(sock))
    #write data out to file
    return(list(optim_sa_results = optim_sa_results_full,mle_summary = full_mle_data))
}

###############################  END KLD FUNCTION   ################################

# OPSII Parameter Optimization

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSII  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)

## run again with best params from first iteration, see if it finds anything better
#init_params <- c(0.06,  0.33,  2.44,  2.61, -0.08,  0.01,  0.99, -0.06,  0.02)
bouts <- ops2_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_ops2_data_cleaned$subjectid)[1:8] #unique(nback_ops2_data_cleaned$subjectid)[1:2]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSII",length(participants))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS2_OptimSA_results_list_1) <- participants

OPS2_OptimSA_results_1 <- map(OPS2_OptimSA_results_list_1, "optim_sa_results")
OPS2_MLE_summaries_1 <- map_dfr(OPS2_OptimSA_results_list_1, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
OPS2_OptimSA_results_list_1

In [ ]:
OPS2_MLE_summaries_1

In [ ]:
OPS2_OptimSA_results_1

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_MLEresults_p1-8.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS2_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_OptimSAtrace_p1-8.csv" #replace 'test' with 'p1-10'
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS2_MLE_summaries_1 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSII_fullMLEdata_p1-8.csv" #replace 'test' with 'p1-10'
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING   #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops2_bouts

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 9-16 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants_2 <- unique(nback_ops2_data_cleaned$subjectid)[9:16] #unique(nback_ops2_data_cleaned$subjectid)[1:2]
participants_2

# Assign unique ports
base_port <- 9850
ports_2 <- base_port + seq_along(participants_2) - 1
ports_2

study_list <- rep("OPSII",length(participants_2))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants_2, port = ports_2), run_sa_for_subject)

names(OPS2_OptimSA_results_list_2) <- participants_2

OPS2_OptimSA_results_2 <- map(OPS2_OptimSA_results_list_2, "optim_sa_results")
OPS2_MLE_summaries_2 <- map_dfr(OPS2_OptimSA_results_list_2, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################


In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_KLresults_p9-16.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS2_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_OptimSAtrace_p9-16.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS2_MLE_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSII_fullKLdata_p9-16.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING   #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops2_bouts

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 17-24 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants_3 <- unique(nback_ops2_data_cleaned$subjectid)[17:24] #unique(nback_ops2_data_cleaned$subjectid)[1:2]
participants_3


# Assign unique ports
base_port <- 9850
ports_3 <- base_port + seq_along(participants_3) - 1
ports_3

study_list <- rep("OPSII",length(participants_3))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_3 <- future_pmap(list(studyid = study_list, pid = participants_3, port = ports_3), run_sa_for_subject)

names(OPS2_OptimSA_results_list_3) <- participants_3

OPS2_OptimSA_results_3 <- map(OPS2_OptimSA_results_list_3, "optim_sa_results")
OPS2_MLE_summaries_3 <- map_dfr(OPS2_OptimSA_results_list_3, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################


In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_KLresults_p17-24.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS2_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_OptimSAtrace_p17-24.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS2_MLE_summaries_3 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSII_fullKLdata_p17-24.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING   #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops2_bouts

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 25-32 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants_4 <- unique(nback_ops2_data_cleaned$subjectid)[25:32] #unique(nback_ops2_data_cleaned$subjectid)[1:2]
participants_4

# Assign unique ports
base_port <- 9850
ports_4 <- base_port + seq_along(participants_4) - 1
ports_4

study_list <- rep("OPSII",length(participants_4))

# Run in parallel across all subjects
OPS2_OptimSA_results_list_4 <- future_pmap(list(studyid = study_list, pid = participants_4, port = ports_4), run_sa_for_subject)

names(OPS2_OptimSA_results_list_4) <- participants_4

OPS2_OptimSA_results_4 <- map(OPS2_OptimSA_results_list_4, "optim_sa_results")
OPS2_MLE_summaries_4 <- map_dfr(OPS2_OptimSA_results_list_4, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################


In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS2_OptimSA_results_4, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_KLresults_p25-32.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS2_OptimSA_results_4, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSII_OptimSAtrace_p25-32.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS2_MLE_summaries_4 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSII_fullKLdata_p25-32.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# OPSI Parameter Optimization

In [ ]:
length(unique(nback_ops1_data_cleaned$subjectid))
unique(nback_ops1_data_cleaned$subjectid)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops1_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_ops1_data_cleaned$subjectid)[1:10]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_1) <- participants

OPS1_OptimSA_results_1 <- map(OPS1_OptimSA_results_list_1, "optim_sa_results")
OPS1_MLE_summaries_1 <- map_dfr(OPS1_OptimSA_results_list_1, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_MLEresults_p1-10.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS1_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_OptimSAtrace_p1-10.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS1_MLE_summaries_1 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSI_fullMLEdata_p1-10.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops1_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_ops1_data_cleaned$subjectid)[11:20]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_2) <- participants

OPS1_OptimSA_results_2 <- map(OPS1_OptimSA_results_list_2, "optim_sa_results")
OPS1_MLE_summaries_2 <- map_dfr(OPS1_OptimSA_results_list_2, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_MLEresults_p11-20.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS1_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_OptimSAtrace_p11-20.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS1_MLE_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSI_fullMLEdata_p11-20.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops1_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_ops1_data_cleaned$subjectid)[21:30]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_3 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_3) <- participants

OPS1_OptimSA_results_3 <- map(OPS1_OptimSA_results_list_3, "optim_sa_results")
OPS1_MLE_summaries_3 <- map_dfr(OPS1_OptimSA_results_list_3, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_MLEresults_p21-30.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS1_OptimSA_results_3, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_OptimSAtrace_p21-30.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS1_MLE_summaries_3 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSI_fullMLEdata_p21-30.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops1_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_ops1_data_cleaned$subjectid)[31:40]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_4 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_4) <- participants

OPS1_OptimSA_results_4 <- map(OPS1_OptimSA_results_list_4, "optim_sa_results")
OPS1_MLE_summaries_4 <- map_dfr(OPS1_OptimSA_results_list_4, "mle_summary")

plan(sequential) 
# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_4, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_MLEresults_p31-40.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS1_OptimSA_results_4, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_OptimSAtrace_p31-40.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS1_MLE_summaries_4 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSI_fullMLEdata_p31-40.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  OPSI  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- ops1_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_ops1_data_cleaned$subjectid)[41:50]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("OPSI",length(participants))

# Run in parallel across all subjects
OPS1_OptimSA_results_list_5 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(OPS1_OptimSA_results_list_5) <- participants

OPS1_OptimSA_results_5 <- map(OPS1_OptimSA_results_list_5, "optim_sa_results")
OPS1_MLE_summaries_5 <- map_dfr(OPS1_OptimSA_results_list_5, "mle_summary")

plan(sequential) 
# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(OPS1_OptimSA_results_5, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_MLEresults_p41-50.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(OPS1_OptimSA_results_5, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_OPSI_OptimSAtrace_p41-50.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- OPS1_MLE_summaries_5 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_OPSI_fullMLEdata_p41-50.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# SERE Parameter Optimization

In [ ]:
## NO SERE for N-BACK

# HIGH Parameter Optimization

In [ ]:
length(unique(nback_high_data_cleaned$subjectid))

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  HIGH  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- high_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_high_data_cleaned$subjectid)[1:9]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("HIGH",length(participants))

# Run in parallel across all subjects
HIGH_OptimSA_results_list_1 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(HIGH_OptimSA_results_list_1) <- participants

HIGH_OptimSA_results_1 <- map(HIGH_OptimSA_results_list_1, "optim_sa_results")
HIGH_MLE_summaries_1 <- map_dfr(HIGH_OptimSA_results_list_1, "mle_summary")

plan(sequential) 

# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
HIGH_OptimSA_results_list_1

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(HIGH_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_HIGH_MLEresults_p1-9.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(HIGH_OptimSA_results_1, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_HIGH_OptimSAtrace_p1-9.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- HIGH_MLE_summaries_1 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_HIGH_fullMLEdata_p1-9.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
#######################   BEGIN Parallel OptimSA SIMULATED ANNEALING  HIGH  #######################
##
## Optimization: Simulated Annealing
## fun = rmse
## We want to use simulated annealing to find the minimum rmse estimates of the parameters...
##

## Create initial parameter values from which to start simulated annealing
## Vector of values in order of:
## c(dat, egs, ut, iu, utmc, utbmc, fpdec, fpmc, fpbmc)
#init_params <- c(0.05,0.25,2.2,2.6,-0.01,0.01,0.98,-0.05,0.02)
bouts <- high_bouts

## Create lower and upper bounds of parameters
#par_lo <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0)
#par_hi <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04)

iterations <- 25
t0 <- 1000
nlimit <- 40

fix_procedural = TRUE

if (fix_procedural) {
    ## run again with best params from first iteration, see if it finds anything better
    #init_params <- c(-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    #init_params <- c(-0.025,0.97,0.96,0.1,0.44,0.95,0.81)
    #init_params <- c(-0.045,0.98,0.99,0.12,0.51,1.11,0.81)
    #init_params <- c(-0.015,0.93,0.97,0.18,0.41,1.07,0.81)
    init_params <- c(-0.025,0.92,0.97,0.14,0.4,1.12,0.8)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(-0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    } else {
    ## run again with best params from first iteration, see if it finds anything better
    init_params <- c(0.06,0.33,2.44,2.61,-0.08,0.01,0.99,-0.06,0.02,-0.025,0.96,0.98,0.25,0.5,1.0,1.0)
    
    ## Create lower and upper bounds of parameters
    par_low <- c(0.04, 0.1, 2.0, 2.0, -0.1, 0.0, 0.96, -0.1, 0.0, -0.05, 0.9, 0.96, 0.1,0.4,0.8,0.8)
    par_high <- c(0.06, 0.4, 3.0, 3.0, 0.0, 0.04, 1.0, 0.0, 0.04, 0.0, 1.0, 1.0, 0.4,0.6,1.2,1.2)
    }

######################################### PARTICIPANTS 1-10 #####################################################
plan(multisession)#, workers = 30)  # Use multicore on Linux/macOS or multisession on Windows

# Subject IDs
participants <- unique(nback_high_data_cleaned$subjectid)[10:17]
participants

# Assign unique ports
base_port <- 9850
ports <- base_port + seq_along(participants) - 1
ports

study_list <- rep("HIGH",length(participants))

# Run in parallel across all subjects
HIGH_OptimSA_results_list_2 <- future_pmap(list(studyid = study_list, pid = participants, port = ports), run_sa_for_subject)

names(HIGH_OptimSA_results_list_2) <- participants

HIGH_OptimSA_results_2 <- map(HIGH_OptimSA_results_list_2, "optim_sa_results")
HIGH_MLE_summaries_2 <- map_dfr(HIGH_OptimSA_results_list_2, "mle_summary")

plan(sequential) 


# ########################   END Parallel OptimSA SIMULATED ANNEALING   ########################

In [ ]:
#df <- map_df(names(OptimSA_results), function(name) {
#  tibble(subjectid = name, phase = OptimSA_results[[name]]$phase, bout = OptimSA_results[[name]]$bout, kl_val = OptimSA_results[[name]]$function_value, par = list(round(OptimSA_results[[name]]$par,2)))
#})

df <- imap_dfr(HIGH_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {
    tibble(
      study     = res$study,
      subjectid = subject,
      phase     = res$phase,
      study_event_id = res$study_event_id,
      bout      = res$bout,
      kl_val    = res$function_value,
      par       = list(c(res$pvt_params,round(res$par, 3)))
    )
  })
})

head(df)
df_long <- df %>% unnest_longer(par, values_to = "value", indices_to = "par_name") %>%
  mutate(par_name = recode(par_name,"1"="dat","2"="egs","3"="ut","4"="iu","5"="utmc","6"="utbmc","7"="fpdec","8"="fpmc","9"="fpbmc","10"="fdbmc","11"="fdc","12"="fddec","13"="ans","14"="bll","15"="mp","16"="lf"))
head(df_long)

df_wide <- df %>% unnest_wider(par, names_sep = "_") %>%
    rename(dat = par_1, egs = par_2, ut=par_3, iu=par_4, utmc=par_5, utbmc=par_6, fpdec=par_7, fpmc=par_8, fpbmc=par_9, fdbmc=par_10, fdc=par_11, fddec=par_12, ans=par_13, bll=par_14, mp=par_15, lf=par_16)
head(df_wide)

In [ ]:
filename <- "NBack_MLE_Individual_HIGH_MLEresults_p10-17.csv"
write.csv(df_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
full_trace_names <- c(
  "n_outer", "loss",
  "x_1", "x_2", "x_3", "x_4", "x_5", "x_6", "x_7", "x_8", "x_9",
  "x_10", "x_11", "x_12", "x_13", "x_14", "x_15", "x_16",
  "n_inner", "temperature", "goodcounter",
  "rf_1", "rf_2", "rf_3", "rf_4", "rf_5", "rf_6", "rf_7",
  "rf_8", "rf_9", "rf_10", "rf_11", "rf_12", "rf_13", "rf_14", "rf_15", "rf_16"
)

trace <- imap_dfr(HIGH_OptimSA_results_2, function(phase_list, subject) {
  imap_dfr(phase_list, function(res, phase) {

    tr <- as.data.frame(res$trace)
    if (sum(grepl("^x_", names(tr))) == 7) {
        names(tr)[names(tr) %in% paste0("x_", 1:7)] <- paste0("x_", 10:16)
        names(tr)[names(tr) %in% paste0("rf_", 1:7)] <- paste0("rf_", 10:16)
    }
    missing_cols <- setdiff(full_trace_names, names(tr))
    if (length(missing_cols) > 0) tr[missing_cols] <- NA
    tr <- tr[, full_trace_names, drop = FALSE]

    tibble(
      study = res$study,
      subjectid = subject,
      phase = res$phase,
      study_event_id = res$study_event_id,
      bout = res$bout,
      trace = list(tr)
    )
  })
})

trace_wide <- trace %>%
  unnest(trace)

trace_wide

In [ ]:
filename <- "NBack_MLE_Individual_HIGH_OptimSAtrace_p10-17.csv"
write.csv(trace_wide,paste("./",filename, sep=""), row.names = FALSE)

In [ ]:
MLE_summaries_wide <- HIGH_MLE_summaries_2 %>% unnest_wider(params, names_sep = "_") %>%
    rename(dat = params_1, egs = params_2, ut=params_3, iu=params_4, utmc=params_5, utbmc=params_6, fpdec=params_7, fpmc=params_8, fpbmc=params_9, fdbmc=params_10, fdc=params_11, fddec=params_12, ans=params_13, bll=params_14, mp=params_15, lf=params_16)
MLE_summaries_wide

In [ ]:
head(MLE_summaries_wide)
filename <- "NBack_MLE_Individual_HIGH_fullMLEdata_p10-17.csv"
write.csv(MLE_summaries_wide,paste("./",filename, sep=""), row.names = FALSE)

# Combine all datasets and save to file

In [ ]:
# Base path
path <- "./"

# List filenames with filepaths
files <- file.path(path, c(
    "NBack_MLE_Individual_OPSI_MLEresults_p1-10.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p11-20.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p21-30.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p31-40.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p41-50.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p1-8.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p9-16.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p17-24.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p25-32.csv",
    "PVT_KL_Individual_SERE_KLresults_p1-9.csv",
    "PVT_KL_Individual_SERE_KLresults_p10-18.csv",
    "PVT_KL_Individual_SERE_KLresults_p19-27.csv",
    "PVT_KL_Individual_SERE_KLresults_p28-36.csv",
    "PVT_KL_Individual_SERE_KLresults_p37-45.csv",
    "PVT_KL_Individual_SERE_KLresults_p46-54.csv",
    "PVT_KL_Individual_SERE_KLresults_p55-63.csv"))


# Function to read and standardize each file
read_and_fix <- function(f) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  
  # Force these columns to character if present
  char_cols <- c("study", "subjectid", "phase", "study_event_id")
  present_char_cols <- intersect(char_cols, names(df))
  df[present_char_cols] <- lapply(df[present_char_cols], as.character)
  
  # Force bout to numeric if present
  if ("bout" %in% names(df)) {
    df$bout <- as.numeric(df$bout)
  }
  
  # Force all other columns to numeric
  other_cols <- setdiff(names(df), c(char_cols, "bout"))
  df[other_cols] <- lapply(df[other_cols], as.numeric)
  
  df
}

# Read files
df_list <- lapply(files, read_and_fix)

# Combine data
combined_df <- bind_rows(df_list)

# Sort data by study, subjectid, phase (Stress, Baseline, Recovery)
# Ensure "phase" is an ordered factor
combined_df$phase <- factor(
  combined_df$phase,
  levels = c("Baseline", "Stress", "Recovery"),
  ordered = TRUE
)

# Order the dataframe
combined_df <- combined_df[
  order(
    combined_df$study,
    combined_df$subjectid,
    combined_df$phase
  ),
]

# Write combined dataset file
write.csv(combined_df, "NBack_MLE_Individual_MLEresults_FullData.csv", row.names = FALSE)
# Save to the same folder
write.csv(
  combined_df,
  file.path(path, "ROPER_CompuCognome_FullData.csv"),
  row.names = FALSE
)


In [ ]:
# Base path
path <- "./"

# List filenames with filepaths
files <- file.path(path, c(
    "NBack_MLE_Individual_OPSI_MLEresults_p1-10.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p11-20.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p21-30.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p31-40.csv",
    "NBack_MLE_Individual_OPSI_MLEresults_p41-50.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p1-8.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p9-16.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p17-24.csv",
    "NBack_MLE_Individual_OPSII_MLEresults_p25-32.csv",
    "PVT_KL_Individual_SERE_KLresults_p1-9.csv",
    "PVT_KL_Individual_SERE_KLresults_p10-18.csv",
    "PVT_KL_Individual_SERE_KLresults_p19-27.csv",
    "PVT_KL_Individual_SERE_KLresults_p28-36.csv",
    "PVT_KL_Individual_SERE_KLresults_p37-45.csv",
    "PVT_KL_Individual_SERE_KLresults_p46-54.csv",
    "PVT_KL_Individual_SERE_KLresults_p55-63.csv",
    "NBack_MLE_Individual_HIGH_MLEresults_p1-9.csv",
    "NBack_MLE_Individual_HIGH_MLEresults_p10-17.csv"))

# Function to read and standardize each file
read_and_fix <- function(f) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  
  # Force these columns to character if present
  char_cols <- c("study", "subjectid", "phase", "study_event_id")
  present_char_cols <- intersect(char_cols, names(df))
  df[present_char_cols] <- lapply(df[present_char_cols], as.character)
  
  # Force bout to numeric if present
  if ("bout" %in% names(df)) {
    df$bout <- as.numeric(df$bout)
  }
  
  # Force all other columns to numeric
  other_cols <- setdiff(names(df), c(char_cols, "bout"))
  df[other_cols] <- lapply(df[other_cols], as.numeric)
  
  df
}

# Read files
df_list <- lapply(files, read_and_fix)

# Combine data
combined_df <- bind_rows(df_list)

# Sort data by study, subjectid, phase (Stress, Baseline, Recovery)
# Ensure "phase" is an ordered factor
combined_df$phase <- factor(
  combined_df$phase,
  levels = c("Baseline", "Stress", "Recovery"),
  ordered = TRUE
)

combined_df$study <- factor(
  combined_df$study,
  levels = c("OPSI", "OPSII", "SERE","HIGH"),
  ordered = TRUE
)

# Order the dataframe
combined_df <- combined_df[
  order(
    combined_df$study,
    combined_df$subjectid,
    combined_df$phase, 
    combined_df$study_event_id
  ),
]

# Save to the same folder
write.csv(
  combined_df,
  file.path(path, "ROPER_CompuCognome_FullData_AllStudies.csv"),
  row.names = FALSE
)


In [ ]:
# Base path
path <- "./"

# Files to combine
files <- file.path(path, c(
  "NBack_MLE_Individual_OPSI_MLEresults_p1-10.csv",
  "NBack_MLE_Individual_OPSI_MLEresults_p11-20.csv",
  "NBack_MLE_Individual_OPSI_MLEresults_p21-30.csv",
  "NBack_MLE_Individual_OPSI_MLEresults_p31-40.csv",
  "NBack_MLE_Individual_OPSI_MLEresults_p41-50.csv",
  "NBack_MLE_Individual_OPSII_MLEresults_p1-8.csv",
  "NBack_MLE_Individual_OPSII_MLEresults_p9-16.csv",
  "NBack_MLE_Individual_OPSII_MLEresults_p17-24.csv",
  "NBack_MLE_Individual_OPSII_MLEresults_p25-32.csv",
    "NBack_MLE_Individual_HIGH_MLEresults_p1-9.csv",
    "NBack_MLE_Individual_HIGH_MLEresults_p10-17.csv",
  "PVT_KL_Individual_OPSI_KLresults_p1-12.csv",
    "PVT_KL_Individual_OPSI_KLresults_p13-24.csv",
    "PVT_KL_Individual_OPSI_KLresults_p25-36.csv",
    "PVT_KL_Individual_OPSI_KLresults_p37-48.csv",
    "PVT_KL_Individual_OPSII_KLresults_p1-10.csv",
    "PVT_KL_Individual_OPSII_KLresults_p11-20.csv",
    "PVT_KL_Individual_OPSII_KLresults_p21-30.csv",
    "PVT_KL_Individual_SERE_KLresults_p1-9.csv",
    "PVT_KL_Individual_SERE_KLresults_p10-18.csv",
    "PVT_KL_Individual_SERE_KLresults_p19-27.csv",
    "PVT_KL_Individual_SERE_KLresults_p28-36.csv",
    "PVT_KL_Individual_SERE_KLresults_p37-45.csv",
    "PVT_KL_Individual_SERE_KLresults_p46-54.csv",
    "PVT_KL_Individual_SERE_KLresults_p55-63.csv",
    "PVT_KL_Individual_HIGH_KLresults_p1-8.csv",
    "PVT_KL_Individual_HIGH_KLresults_p9-15.csv"
))

# Key columns that identify a row
key_cols <- c("study", "subjectid", "phase", "study_event_id")

# Shared parameter columns present in both task types
shared_param_cols <- c(
  "dat", "egs", "ut", "iu", "utmc", "utbmc", "fpdec", "fpmc", "fpbmc"
)

# NBack-only parameter columns
nback_only_cols <- c(
  "fdbmc", "fdc", "fddec", "ans", "bll", "mp", "lf"
)

# Function to read and standardize each file
read_and_fix <- function(f) {
  df <- read.csv(f, stringsAsFactors = FALSE)

  # Force key character columns
  char_cols <- intersect(c("study", "subjectid", "phase", "study_event_id"), names(df))
  df[char_cols] <- lapply(df[char_cols], as.character)

  # bout numeric
  if ("bout" %in% names(df)) {
    df$bout <- as.numeric(df$bout)
  }

  # All non-key/non-bout columns numeric
  other_cols <- setdiff(names(df), c("study", "subjectid", "phase", "study_event_id", "bout"))
  df[other_cols] <- lapply(df[other_cols], as.numeric)

  # Track source task from filename
  df$task <- ifelse(grepl("PVT", basename(f), ignore.case = TRUE), "PVT", "NBack")

  df
}

# Read all files
df_list <- lapply(files, read_and_fix)

# Split into PVT and NBack
pvt_df <- bind_rows(df_list[df_list |> sapply(function(x) unique(x$task) == "PVT")]) |>
  select(-task)

nback_df <- bind_rows(df_list[df_list |> sapply(function(x) unique(x$task) == "NBack")]) |>
  select(-task)

# Rename NBack kl_val -> mle_val
if ("kl_val" %in% names(nback_df)) {
  nback_df <- nback_df |> rename(mle_val = kl_val)
}

# Make sure PVT has mle_val column if absent
if (!"mle_val" %in% names(pvt_df)) {
  pvt_df$mle_val <- NA_real_
}

# Make sure NBack has kl_val column if absent
if (!"kl_val" %in% names(nback_df)) {
  nback_df$kl_val <- NA_real_
}

# Full join without bout in the key
merged_df <- full_join(
  pvt_df,
  nback_df,
  by = key_cols,
  suffix = c("_pvt", "_nb")
)

# Use NBack bout when present; otherwise PVT bout
merged_df$bout <- dplyr::coalesce(merged_df$bout_nb, merged_df$bout_pvt)

# Coalesce shared parameters
for (col in shared_param_cols) {
  pvt_col <- paste0(col, "_pvt")
  nb_col  <- paste0(col, "_nb")
  merged_df[[col]] <- dplyr::coalesce(merged_df[[pvt_col]], merged_df[[nb_col]])
}

# Keep PVT kl_val and NBack mle_val
merged_df$kl_val  <- dplyr::coalesce(merged_df$kl_val_pvt, merged_df$kl_val_nb)
merged_df$mle_val <- dplyr::coalesce(merged_df$mle_val_pvt, merged_df$mle_val_nb)

# Final dataset
final_df <- merged_df |>
  select(
    all_of(key_cols),
    bout,
    kl_val,
    mle_val,
    all_of(shared_param_cols),
    any_of(nback_only_cols)
  )

# Sort phase in desired order
final_df$phase <- factor(
  final_df$phase,
  levels = c("Baseline", "Stress", "Recovery"),
  ordered = TRUE
)

final_df <- final_df |>
  arrange(study, subjectid, phase, study_event_id, bout)

# Save
write.csv(
  final_df,
  file.path(path, "ROPER_CompuCognome_FullData_AllStudies_Merged.csv"),
  row.names = FALSE
)